# Libraries and Conf

In [ ]:
# !pip install crate
# import crate

import os
import math
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Preprocessing Files

## Utility Functions

In [ ]:
#taken from https://www.geeksforgeeks.org/sum-two-large-numbers/
#multiply long numbers
def multiply(num1, num2):
    num1 = str(num1)
    num2 = str(num2)

    len1 = len(num1)
    len2 = len(num2)
    if len1 == 0 or len2 == 0:
        return "0"

    # will keep the result number in vector
    # in reverse order
    result = [0] * (len1 + len2)
    
    # Below two indexes are used to
    # find positions in result.
    i_n1 = 0
    i_n2 = 0

    # Go from right to left in num1
    for i in range(len1 - 1, -1, -1):
        carry = 0
        n1 = ord(num1[i]) - 48

        # To shift position to left after every
        # multiplication of a digit in num2
        i_n2 = 0

        # Go from right to left in num2
        for j in range(len2 - 1, -1, -1):
            
            # Take current digit of second number
            n2 = ord(num2[j]) - 48
        
            # Multiply with current digit of first number
            # and add result to previously stored result
            # at current position.
            summ = n1 * n2 + result[i_n1 + i_n2] + carry

            # Carry for next iteration
            carry = summ // 10

            # Store result
            result[i_n1 + i_n2] = summ % 10

            i_n2 += 1

            # store carry in next cell
        if (carry > 0):
            result[i_n1 + i_n2] += carry

            # To shift position to left after every
            # multiplication of a digit in num1.
        i_n1 += 1
        
        # print(result)

    # ignore '0's from the right
    i = len(result) - 1
    while (i >= 0 and result[i] == 0):
        i -= 1

    # If all were '0's - means either both or
    # one of num1 or num2 were '0'
    if (i == -1):
        return "0"

    # generate the result string
    s = ""
    while (i >= 0):
        s += chr(result[i] + 48)
        i -= 1

    return s

#get engagement of tweet
def calculate_engagement(retweet_count, like_count, quote_count, reply_count, user_followers_count):    
    if retweet_count != None and like_count != None and quote_count != None and reply_count != None and user_followers_count != None:
        engagement = 0
        if not math.isnan(retweet_count) and not math.isnan(like_count) and not math.isnan(quote_count) and not math.isnan(reply_count) and not math.isnan(user_followers_count):
            retweet_count = int(retweet_count)
            like_count = int(like_count)
            quote_count = int(quote_count)
            reply_count = int(reply_count)
            user_followers_count = int(user_followers_count)

            if user_followers_count > 0:
                engagement = ((retweet_count + like_count + quote_count + reply_count) / user_followers_count) * 100
        return engagement

#get extended reach of tweet
def calculate_extendedReach(retweet_count, user_tweet_count):
    if retweet_count is not None and user_tweet_count is not None:
        extendedReach = 0
        if not math.isnan(retweet_count) and not math.isnan(user_tweet_count):
            retweet_count = int(retweet_count)
            user_tweet_count = int(user_tweet_count)

            if user_tweet_count > 0:
                extendedReach = (retweet_count / user_tweet_count) * 100
        return extendedReach

#get impressions of tweet
def calculate_impressions(user_followers_count, user_tweet_count):
    if user_followers_count is not None and user_tweet_count is not None:
        if not math.isnan(user_followers_count) and not math.isnan(user_tweet_count):
            user_followers_count = int(user_followers_count)
            user_tweet_count = int(user_tweet_count)
            
            impressions = multiply(user_followers_count, user_tweet_count)
            return impressions

#get the day and time features from datetime
def extract_day_time(date_string):
    if isinstance(date_string, str):
        date_parts = date_string.split(' ')

        if len(date_parts) >= 2:
            day = date_parts[0]
            time = date_parts[1][:8]
            return (day, time)

#get new extracted features for further analysis
def extract_new_fatures(df):
    df[['day', 'time']] = df['date'].apply(lambda date : extract_day_time(date)).apply(pd.Series)
    df['engagement_rate'] = df.apply(lambda row: calculate_engagement(row['tweet_retweet_count'], row['tweet_favorite_count'], 
                                                                          row['tweet_quote_count'], row['tweet_reply_count'], 
                                                                          row['user_followers_count']), axis=1)

    df['extended_reach'] = df.apply(lambda row: calculate_extendedReach(row['tweet_retweet_count'], row['user_statuses_count']), axis=1)

    df['possible_impressions'] = df.apply(lambda row: calculate_impressions(row['user_followers_count'], row['user_statuses_count']), axis=1)
    return df  

# def rename_columns(df):

#     df.rename(columns={'tweet_text':'text', 'tweet_retweet_count': 'retweet_count', 
#                          'tweet_favorite_count': 'like_count', 'tweet_quote_count': 'quote_count', 
#                          'tweet_reply_count': 'reply_count', 'user_followers_count': 'author_followers_count', 
#                         'user_statuses_count': 'author_tweet_count'}, inplace=True)
#     return df

In [ ]:
def preprocess_network(path, desPath):  
    
    total = 0
    for folder in os.listdir(path):
        newPath = path + "/" + folder
        print("Processing ", folder, ".....")

        count = 0
        for fileName in os.listdir(newPath):
            if fileName != ".ipynb_checkpoints":
                data = pd.read_csv(newPath + "/" + fileName, encoding="utf-8", lineterminator='\n')
                if not data.empty:
                    data = extract_new_fatures(data)
                    data['network_of'] = str(folder)  
                    data['user_group'] = path.split("-")[0]
                    data.dropna(subset=['day', 'time', 'possible_impressions'], inplace=True)
                    
                    data.to_csv(desPath + "/" + str(data.user_screen_name[0]) + "_" + str(folder) + ".csv", index=False)
                    count += 1
                    total += 1
        print("Added ", count, " users.....")

    print("--------------------" + "Done Processing " + path + " with " + str(total) + " files--------------------")

# preprocess_network("control-netwrok-raw", "English/user-network/control-network")
# preprocess_network("diagnosed-netwrok-raw", "English/user-network/diagnosed-network")


# Feature Extraction

## Utility Functions

In [ ]:
def avg_score_cal(df):
    
    """
    Calculates the average like, reply etc. score of a user
    
    input: takes a dataframe containing tweets of a user
    output: returns the average like, reply, retweet, emojis, sentiment and engagement rate of the tweets from a user
    """
    
    total_tweets = df.shape[0]
    avg_tweet_favorite_count = df.tweet_favorite_count.sum()/total_tweets
    avg_tweet_retweet_count = df.tweet_retweet_count.sum()/total_tweets
    avg_tweet_reply_count = df.tweet_reply_count.sum()/total_tweets
    avg_engagement_rate = df.engagement_rate.sum()/total_tweets
    avg_tweets_per_day = total_tweets/len(df.day.value_counts().values)

    
    return avg_tweets_per_day, avg_tweet_favorite_count, avg_tweet_retweet_count, avg_tweet_reply_count, avg_engagement_rate

# https://stackoverflow.com/a/70018607/11105356

def time_range(time):
    
    """
    Calculates the period of the day based on time
    
    input: time of the day as string in hour:minute:second format
    output: returns the period of the day as string 
    
    """
    
    hour = datetime.strptime(time, '%H:%M:%S').hour

    if hour > 20:
      return "Night" # 21-23
    elif hour > 16:
      return "Evening" # 17-20
    elif hour > 11:
      return "Afternoon" # 12-16
    elif hour > 4:
      return "Morning" # 5-11
    else:
      return "Midnight" # 1-4


def day_of_week(day):
    
    """
    Calculates the day of the week based on date
    
    input: date as string in YYYY:MM:DD format
    output: returns the day of the week as string 
    
    """
    dt = datetime.strptime(day, '%Y-%m-%d')
    day_of_week = dt.strftime('%A')
    
    return day_of_week


def most_active_day_cal(df):
    
    """
    Calculates which day of the week the user is most active 
    
    input: takes a dataframe containing tweets of a user
    output: returns the most active day of the week for a user
    """
    df["day_of_week"] = df.day.apply(lambda day: day_of_week(day))
    most_active_day = df.day_of_week.value_counts().idxmax()
    
    return most_active_day
    

def most_active_period_cal(df):
    
    """
    Calculates when the majority of the tweets were posted by a user
    
    input: takes a dataframe containing tweets of a user
    output: returns the most active period of the day for a user
    """
    
    df["time_of_the_day"] = df.time.apply(lambda time: time_range(time))
    most_active_period = df.time_of_the_day.value_counts().idxmax()
    
    return most_active_period


In [ ]:
path = "English/user-network"
user_list = pd.DataFrame(columns = ["userfile_name", "user_id", "user_screen_name", "user_followers_count", "user_friends_count", 
                                    "user_statuses_count", "possible_impressions", "avg_tweets_per_day", 
                                    "avg_tweet_favorite_count", "avg_tweet_reply_count", "avg_tweet_retweet_count", 
                                    "avg_engagement_rate", "most_active_period", "most_active_day", "network_of", "user_group"])

for folder in os.listdir(path):
    newPath = path + "/" + folder
    print("Processing ", folder, ".....")
    for fileName in os.listdir(newPath):
        data = pd.read_csv(newPath + "/" + fileName)

        userfile_name = fileName.split('.csv')[0]
        user_id = data['user_id'].iloc[0]
        user_screen_name = data['user_screen_name'].iloc[0]
        user_followers_count = data['user_followers_count'].iloc[0]
        user_friends_count = data['user_friends_count'].iloc[0]
        user_statuses_count = data['user_statuses_count'].iloc[0]
        possible_impressions = data['possible_impressions'].iloc[0]
        user_group = data['user_group'].iloc[0]
        network_of = data['network_of'].iloc[0]
        
        avg_tweets_per_day, avg_tweet_favorite_count, avg_tweet_retweet_count, avg_tweet_reply_count, avg_engagement_rate = avg_score_cal(data)
        most_active_period = most_active_period_cal(data)
        most_active_day = most_active_day_cal(data)
        
        

        user_list = user_list.append({'userfile_name' : userfile_name, 'user_id' : user_id, 'user_screen_name' : user_screen_name,
                          'user_followers_count' : user_followers_count, 'user_friends_count' : user_friends_count, 
                          'user_statuses_count' : user_statuses_count,  
                          'possible_impressions' : possible_impressions, "avg_tweets_per_day" : avg_tweets_per_day,
                          'avg_tweet_favorite_count' : avg_tweet_favorite_count, 'avg_tweet_reply_count': avg_tweet_reply_count, 
                          "avg_tweet_retweet_count" : avg_tweet_retweet_count, 
                          'avg_engagement_rate' : avg_engagement_rate, 'most_active_period' : most_active_period, 
                          'most_active_day' : most_active_day, 'network_of' : network_of, "user_group" : user_group}, 
                    ignore_index = True)

user_list.to_csv('English/network_features.csv', index=False)

In [ ]:
data = pd.read_csv("English/network_features.csv")
data

In [ ]:
# import csv
# from crate import client

# connection = client.connect("http://129.128.215.129:4200/", username="crate")

# cursor = connection.cursor()

# batch_size = 1000

# cursor.execute('SELECT * FROM twitter-network LIMIT 1;')
# total_rows = cursor.fetchone()[0]

# with open('energy_east_tweets2.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow([i[0] for i in cursor.description])
#     for offset in range(0, 10000, batch_size):
#         cursor.execute(f'SELECT * FROM twitter-network LIMIT {batch_size} OFFSET {offset};')
#         rows = cursor.fetchall()
#         for row in rows:
#             writer.writerow(row)

# cursor.close()
# connection.close()